# Phase 2b: Controlled Experiments to Resolve Weak Classes

This notebook executes controlled experiments targeting the weak defect classes (**crazing** and **rolled-in scale**) discovered during the Phase 2a baseline evaluation:
- **E0 (Baseline)**: YOLOv8s at imgsz=640 (control benchmark)
- **E1 (Resolution)**: YOLOv8s at imgsz=320 (speed/accuracy trade-off)
- **E2 (Model Capacity)**: YOLOv8m at imgsz=640 (25.9M params vs 11.2M params)
- **E3 (Domain Augmentation)**: YOLOv8s at imgsz=640 with steel texture augmentations (flips, HSV brightness/contrast, small rotation, mixup off, close_mosaic=10)

All experiments are trained on GPU, evaluated strictly on the validation partition (216 images), and exported to `reports/phase2b/` with `comparison.csv` and grouped per-class bar charts.


> **Smoke Test Flag**: `SMOKE_TEST = True` trains each experiment for 1 epoch only to validate the entire pipeline (training, evaluation, results.json, comparison.csv, bar chart, zip export) in a few minutes. Set `SMOKE_TEST = False` to run full 50-epoch experiments.


In [ ]:
import os
import subprocess
from pathlib import Path

# --- CONFIGURATION VARIABLES ---
REPO_URL = "https://github.com/AaruneshAP/defect-inspection.git"
BRANCH_NAME = "main"
REPO_NAME = "defect-inspection"
SMOKE_TEST = True

working_dir = Path("/kaggle/working")
if working_dir.exists():
    target_repo = working_dir / REPO_NAME
    if not target_repo.exists():
        subprocess.run(["git", "clone", "-b", BRANCH_NAME, REPO_URL], check=True, cwd=str(working_dir))
    os.chdir(target_repo)
else:
    print("Running in local environment; repository root confirmed.")

subprocess.run(["pip", "install", "-q", "ultralytics", "pyyaml", "matplotlib", "pandas"], check=True)


## 1. Discover NEU-DET Dataset & Prepare YOLO Partitions

Uses `find_kaggle_dataset` to locate the dataset under `/kaggle/input`. Raises an explicit `FileNotFoundError` if unattached.


In [ ]:
from pathlib import Path
from src.data import convert_dataset_to_yolo, find_kaggle_dataset

raw_data_dir = find_kaggle_dataset("/kaggle/input")
print(f"Dataset root identified at: {raw_data_dir.resolve()}")

yolo_data_dir = Path("data/yolo")
if not (yolo_data_dir / "data.yaml").exists():
    print("Converting NEU dataset to YOLO format (85/15 stratified train/val split)...")
    convert_dataset_to_yolo(
        data_dir=str(raw_data_dir),
        output_dir="data/yolo",
        val_ratio=0.15,
        seed=42,
    )
print("YOLO dataset verified at data/yolo/data.yaml")


## 2. Execute Controlled Experiments (E0, E1, E2, E3)

Runs all four configurations sequentially, saving each experiment's `results.json` and metric plots into `reports/phase2b/<experiment>/`.


In [ ]:
import json
import shutil
from pathlib import Path
from src.train import (
    build_comparison_table,
    is_cache_valid,
    load_config,
    plot_per_class_comparison,
    run_training,
)

experiments = [
    ("e0_baseline", "configs/baseline.yaml"),
    ("e1_imgsz320", "configs/e1_imgsz320.yaml"),
    ("e2_yolov8m_640", "configs/e2_yolov8m_640.yaml"),
    ("e3_augmented", "configs/e3_augmented.yaml"),
]

reports_dir = Path("reports/phase2b_smoke") if SMOKE_TEST else Path("reports/phase2b")
reports_dir.mkdir(parents=True, exist_ok=True)

experiment_results = {}

for exp_id, cfg_path in experiments:
    print("=" * 50)
    print(f"Starting Experiment: {exp_id} ({cfg_path})")
    print("=" * 50)

    cfg = load_config(cfg_path)
    exp_target_dir = reports_dir / exp_id
    exp_target_dir.mkdir(parents=True, exist_ok=True)
    cached_json = exp_target_dir / "results.json"

    if cached_json.exists() and is_cache_valid(cached_json, cfg, smoke_test=SMOKE_TEST):
        print(f"Reusing matching cached results for {exp_id} from {cached_json}")
        with open(cached_json, "r", encoding="utf-8") as f:
            experiment_results[exp_id] = json.load(f)
        continue

    train_output = run_training(config_path=cfg_path, project="runs", name=exp_id, smoke_test=SMOKE_TEST)
    run_dir = Path(train_output["run_dir"])

    for fname in ["results.json", "results.csv", "results.png"]:
        src_f = run_dir / fname
        if src_f.exists():
            shutil.copy2(src_f, exp_target_dir / fname)

    val_dir = run_dir / "val"
    if not val_dir.exists():
        val_dir = run_dir
    for fname in ["confusion_matrix_normalized.png", "BoxPR_curve.png"]:
        src_f = val_dir / fname
        if src_f.exists():
            shutil.copy2(src_f, exp_target_dir / fname)

    # Ensure results.json records model, imgsz, seed, epochs, and smoke_test
    results_path = exp_target_dir / "results.json"
    with open(results_path, "r", encoding="utf-8") as f:
        saved_payload = json.load(f)
    saved_payload["experiment"] = exp_id
    saved_payload["smoke_test"] = SMOKE_TEST
    saved_payload["model"] = cfg["model"]
    saved_payload["imgsz"] = cfg["imgsz"]
    saved_payload["seed"] = cfg["seed"]
    saved_payload["epochs"] = 1 if SMOKE_TEST else cfg["epochs"]
    with open(results_path, "w", encoding="utf-8") as f:
        json.dump(saved_payload, f, indent=2)

    experiment_results[exp_id] = saved_payload
    print(f"Finished {exp_id} - mAP50: {saved_payload.get('mAP50', 0.0):.4f}")


## 3. Build Comparison Table & Per-Class AP50 Visualization

Aggregates all experiment outputs into `reports/phase2b/comparison.csv` and renders a grouped bar chart for per-class comparison.


In [ ]:
import pandas as pd
from IPython.display import Image, display

comparison_csv_path = reports_dir / "comparison.csv"
comparison_rows = build_comparison_table(reports_dir, output_csv=comparison_csv_path)

comparison_png_path = reports_dir / "per_class_ap50_comparison.png"
plot_per_class_comparison(comparison_rows, output_png=comparison_png_path)

df = pd.DataFrame(comparison_rows)
print("\nPhase 2b Controlled Experiment Results Summary:")
display_cols = [
    "experiment",
    "model",
    "imgsz",
    "precision",
    "recall",
    "mAP50",
    "mAP50-95",
    "crazing_AP50",
    "rolled-in_scale_AP50",
    "scratches_AP50",
]
available_cols = [c for c in display_cols if c in df.columns]
print(df[available_cols].to_string(index=False))

if comparison_png_path.exists():
    display(Image(str(comparison_png_path)))


## 4. Archive Artifacts for Download

Zips `reports/phase2b` so that all experiment CSVs, JSONs, and plots can be downloaded directly from the Kaggle interface.


In [ ]:
import shutil
from pathlib import Path

working_dir = Path("/kaggle/working")
zip_name = "phase2b_smoke_outputs" if SMOKE_TEST else "phase2b_outputs"
if working_dir.exists():
    shutil.make_archive(str(working_dir / zip_name), "zip", str(reports_dir))
    print(f"Archive ready at {working_dir / zip_name}.zip")
else:
    print(f"Local run complete; reports available at {reports_dir}/")
